# EDA — Регион 58: Пензенская область

Исследование данных о продаже квартир по региону за период 2018–2021.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

REGION_CODE = 58
REGION_NAME = 'Пензенская область'

## 1. Загрузка данных

In [ ]:
df_all = pd.read_csv('../../data/processed/merged_clean.csv', parse_dates=['date'])
df = df_all[df_all['region'] == REGION_CODE].copy()
df['year']     = df['date'].dt.year
df['month']    = df['date'].dt.month
df['price_m2'] = df['price'] / df['area']

print(f'Всего по России: {len(df_all):,}')
print(f'Регион {REGION_CODE} ({REGION_NAME}): {len(df):,} записей ({len(df)/len(df_all)*100:.2f}% от РФ)')
print(f'Период: {df["date"].min().date()} — {df["date"].max().date()}')

## 2. Основная статистика

In [ ]:
median_region    = df['price'].median()
median_russia    = df_all['price'].median()
median_m2_region = df['price_m2'].median()
median_m2_russia = (df_all['price'] / df_all['area']).median()

region_medians = df_all.groupby('region')['price'].median().sort_values(ascending=False)
rank = (list(region_medians.index).index(REGION_CODE) + 1) if REGION_CODE in region_medians.index else None
total_regions = len(region_medians)

sep = '=' * 50
print(sep)
print(f'Медианная цена:    {median_region/1e6:.2f} млн руб.  (РФ: {median_russia/1e6:.2f} млн)')
print(f'Медианная цена/м²: {median_m2_region:,.0f} руб./м²  (РФ: {median_m2_russia:,.0f})')
print(f'Ранг по медиане:   {rank} из {total_regions} регионов')
print(sep)

print('\nСтатистика цен (руб.):')
display(df['price'].describe().rename({'count':'кол-во','mean':'среднее','std':'std','min':'мин','max':'макс'}).to_frame('значение').round(0))

print('\nСтатистика площади, цены/м²:')
display(df[['area', 'kitchen_area', 'price_m2']].describe().round(1))

## 3. Источники данных и качество

In [ ]:
src_counts = df['source'].value_counts()
print('Источники данных:')
for src, cnt in src_counts.items():
    print(f'  {src}: {cnt:,} ({cnt/len(df)*100:.1f}%)')

nan_k = df['kitchen_area'].isna().sum()
print(f'\nПропуски kitchen_area: {nan_k:,} ({nan_k/len(df)*100:.1f}%)')

obj_map = {0: 'Вторичка', 1: 'Новостройка'}
obj_counts = df['object_type'].value_counts().rename(obj_map)
print('\nТип объекта:')
for t, cnt in obj_counts.items():
    print(f'  {t}: {cnt:,} ({cnt/len(df)*100:.1f}%)')

## 4. Динамика цен по месяцам

In [ ]:
monthly = df.groupby('date').agg(
    median_price=('price', 'median'),
    median_price_m2=('price_m2', 'median'),
    count=('price', 'count'),
).reset_index()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(monthly['date'], monthly['median_price'] / 1e6, color='steelblue', linewidth=2)
axes[0].set_ylabel('Медианная цена, млн руб.')
axes[0].set_title(f'Динамика цен — {REGION_NAME}')
axes[0].grid(alpha=0.3)

axes[1].plot(monthly['date'], monthly['median_price_m2'], color='darkorange', linewidth=2)
axes[1].set_ylabel('Цена за м², руб.')
axes[1].grid(alpha=0.3)

axes[2].bar(monthly['date'], monthly['count'], color='steelblue', alpha=0.7, width=20)
axes[2].set_ylabel('Кол-во объявлений')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Сезонность цен (по месяцам года)

In [ ]:
month_names = ['Янв','Фев','Мар','Апр','Май','Июн','Июл','Авг','Сен','Окт','Ноя','Дек']
seasonal    = df.groupby('month')['price'].median() / 1e6
seasonal_m2 = df.groupby('month')['price_m2'].median()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(range(1, 13), seasonal.reindex(range(1,13)).values, color='steelblue', alpha=0.8)
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(month_names)
axes[0].set_ylabel('Медианная цена, млн руб.')
axes[0].set_title('Сезонность — цена')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(range(1, 13), seasonal_m2.reindex(range(1,13)).values, color='darkorange', alpha=0.8)
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(month_names)
axes[1].set_ylabel('Медианная цена/м², руб.')
axes[1].set_title('Сезонность — цена/м²')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Распределение цен

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

price_mln = df['price'] / 1e6
cap = price_mln.quantile(0.99)
axes[0].hist(price_mln[price_mln <= cap], bins=60, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].axvline(price_mln.median(), color='red', linestyle='--',
                label=f'Медиана: {price_mln.median():.2f} млн')
axes[0].set_xlabel('Цена, млн руб.')
axes[0].set_ylabel('Кол-во объявлений')
axes[0].set_title('Распределение цен (до 99 перцентиля)')
axes[0].legend()
axes[0].grid(alpha=0.3)

pm2 = df['price_m2']
cap_m2 = pm2.quantile(0.99)
axes[1].hist(pm2[pm2 <= cap_m2], bins=60, color='darkorange', alpha=0.8, edgecolor='white')
axes[1].axvline(pm2.median(), color='red', linestyle='--',
                label=f'Медиана: {pm2.median():,.0f} руб./м²')
axes[1].set_xlabel('Цена за м², руб.')
axes[1].set_ylabel('Кол-во объявлений')
axes[1].set_title('Распределение цены за м²')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Вторичка vs Новостройка

In [ ]:
obj_map = {0: 'Вторичка', 1: 'Новостройка'}
df['obj_label'] = df['object_type'].map(obj_map)

stats_obj = df.groupby('obj_label').agg(
    count=('price', 'count'),
    median_price=('price', 'median'),
    median_price_m2=('price_m2', 'median'),
    median_area=('area', 'median'),
).round(0)
stats_obj['медиана цены, млн']  = (stats_obj['median_price'] / 1e6).round(2)
stats_obj['медиана цены/м²']    = stats_obj['median_price_m2'].astype(int)
stats_obj['медиана площади, м²'] = stats_obj['median_area']
display(stats_obj[['count', 'медиана цены, млн', 'медиана цены/м²', 'медиана площади, м²']])

dyn = df.groupby(['date', 'obj_label'])['price'].median().unstack() / 1e6
fig, ax = plt.subplots(figsize=(13, 4))
for col, color in zip(dyn.columns, ['steelblue', 'darkorange']):
    if col in dyn.columns:
        ax.plot(dyn.index, dyn[col], label=col, color=color, linewidth=2)
ax.set_ylabel('Медианная цена, млн руб.')
ax.set_title('Динамика: вторичка vs новостройка')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Площадь и комнатность

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

area_cap = df['area'].quantile(0.99)
axes[0].hist(df['area'][df['area'] <= area_cap], bins=60, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].axvline(df['area'].median(), color='red', linestyle='--',
                label=f'Медиана: {df["area"].median():.0f} м²')
axes[0].set_xlabel('Площадь, м²')
axes[0].set_title('Распределение площади')
axes[0].legend()
axes[0].grid(alpha=0.3)

rooms_labels = {0: 'Студия', 1: '1к', 2: '2к', 3: '3к', 4: '4к'}
rooms_cnt = df['rooms'].value_counts().sort_index()
rooms_plot = rooms_cnt[rooms_cnt.index < 5].copy()
ge5 = rooms_cnt[rooms_cnt.index >= 5].sum()
if ge5 > 0:
    rooms_plot[5] = ge5
    rooms_labels[5] = '5к+'
rooms_plot.index = [rooms_labels.get(r, f'{r}к') for r in rooms_plot.index]
axes[1].bar(rooms_plot.index, rooms_plot.values, color='steelblue', alpha=0.8)
axes[1].set_xlabel('Комнатность')
axes[1].set_ylabel('Кол-во объявлений')
axes[1].set_title('Комнатность')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

price_by_rooms = df[df['rooms'] < 6].groupby('rooms')['price'].median() / 1e6
price_by_rooms.index = [rooms_labels.get(r, f'{r}к') for r in price_by_rooms.index]
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(price_by_rooms.index, price_by_rooms.values, color='darkorange', alpha=0.8)
ax.set_ylabel('Медианная цена, млн руб.')
ax.set_title('Медианная цена по комнатности')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Этажность

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

level_cap = df['level'].quantile(0.99)
axes[0].hist(df['level'][df['level'] <= level_cap], bins=40, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Этаж квартиры')
axes[0].set_ylabel('Кол-во объявлений')
axes[0].set_title('Распределение по этажу')
axes[0].grid(alpha=0.3)

levels_cap = df['levels'].quantile(0.99)
axes[1].hist(df['levels'][df['levels'] <= levels_cap], bins=40, color='darkorange', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Этажей в доме')
axes[1].set_ylabel('Кол-во объявлений')
axes[1].set_title('Распределение по этажности дома')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Корреляция площади и цены

In [ ]:
sample = df.sample(min(10_000, len(df)), random_state=42)
area_cap  = sample['area'].quantile(0.99)
price_cap = sample['price'].quantile(0.99)
sample = sample[(sample['area'] <= area_cap) & (sample['price'] <= price_cap)]

corr = df['area'].corr(df['price'])

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(sample['area'], sample['price'] / 1e6, alpha=0.3, s=5, color='steelblue')
ax.set_xlabel('Площадь, м²')
ax.set_ylabel('Цена, млн руб.')
ax.set_title(f'Площадь vs Цена  (Pearson r = {corr:.3f})')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Коэффициент корреляции площади и цены: {corr:.3f}')

## 11. Расстояние до центра региона vs Цена

In [ ]:
sample2   = df.sample(min(10_000, len(df)), random_state=42)
dist_cap  = sample2['dist_to_center'].quantile(0.99)
price_cap2 = sample2['price'].quantile(0.99)
sample2 = sample2[(sample2['dist_to_center'] <= dist_cap) & (sample2['price'] <= price_cap2)]

corr_dist = df['dist_to_center'].corr(df['price'])

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(sample2['dist_to_center'], sample2['price'] / 1e6, alpha=0.3, s=5, color='darkorange')
ax.set_xlabel('Расстояние до центра региона, км')
ax.set_ylabel('Цена, млн руб.')
ax.set_title(f'dist_to_center vs Цена  (Pearson r = {corr_dist:.3f})')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Корреляция dist_to_center и цены: {corr_dist:.3f}')

df['dist_bin'] = pd.cut(df['dist_to_center'],
                        bins=[0, 5, 15, 30, 60, 120, 9999],
                        labels=['0-5', '5-15', '15-30', '30-60', '60-120', '120+'])
dist_price = df.groupby('dist_bin', observed=True)['price'].median() / 1e6
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(dist_price.index.astype(str), dist_price.values, color='darkorange', alpha=0.8)
ax.set_xlabel('Расстояние до центра, км')
ax.set_ylabel('Медианная цена, млн руб.')
ax.set_title('Медианная цена по удалённости от центра')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Сравнение с медианой по России

In [ ]:
top = df_all.groupby('region')['price'].median().sort_values(ascending=False).reset_index()
top['rank']      = range(1, len(top) + 1)
top['price_mln'] = (top['price'] / 1e6).round(2)

region_row = top[top['region'] == REGION_CODE]
print(f'Медианная цена по региону: {median_region/1e6:.2f} млн руб.')
print(f'Медианная цена по России:  {median_russia/1e6:.2f} млн руб.')
print(f'Отношение к медиане РФ:    {median_region/median_russia:.2f}x')
if len(region_row):
    print(f'Место среди регионов:      {region_row.iloc[0]["rank"]} из {len(top)}')

fig, ax = plt.subplots(figsize=(14, 5))
colors_bar = ['darkorange' if r == REGION_CODE else 'steelblue' for r in top['region']]
ax.bar(top['rank'], top['price_mln'], color=colors_bar, alpha=0.8)
ax.axhline(median_russia / 1e6, color='red', linestyle='--', linewidth=1.5,
           label=f'Медиана РФ: {median_russia/1e6:.2f} млн')
ax.set_xlabel('Ранг региона')
ax.set_ylabel('Медианная цена, млн руб.')
ax.set_title(f'Позиция региона среди всех субъектов РФ (оранжевый = {REGION_NAME})')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()